# Radar pit sites: forcing and model setup

A single place to inspect **TKG2, TKG3, C161, TKG4, and DG1** before looking at model results. This notebook only reads the prepared files; rerunning it never regenerates or overwrites model inputs. Use the **JIF snow modeling** kernel.

The raw HRRR-AK and corrected variants cover 2019-10-01 05:00–2026-09-29 04:00 UTC. Temperature alone is corrected using `TA_site = TA_HRRR - 0.006 × (site elevation − HRRR orography)` in K. HRRR orography varies in the stored UTM dataset, so each hour's orography is used. All other forcing fields are identical between variants. One missing source hour (2024-05-23 03:00 UTC) remains nodata in the input files.

TKG2 (1,326 m), TKG3 (1,203 m), C161 (1,483 m), and DG1 (1,018 m) use the pit coordinates. TKG4 retains its existing model point and 1,120 m elevation, which differ from the nearby pit. Pit/radar horizontal datum is provisionally EPSG:32608; vertical datum is unconfirmed. TKG3 has a pit-date discrepancy flag.

All runs use `heatflux_soiltemp_zero`: no soil module, zero geothermal heat flux, and a fixed 273.15 K ground-temperature generator. Initial profiles are empty; glacier ice/firn layers are not supplied. The new sites use flat terrain while TKG4 retains 38°, so differences across sites are not exclusively elevation effects. No figures are exported automatically.

Raw TKG4 and raw DG1 use the documented [empty-column numerical guard](numerics/README.md) for an installed SNOWPACK 3.7.0 vapor-transport crash. The other runs use the original executable directly. Keep this distinction in the run audit when interpreting results.


## Site metadata

Coordinates below locate the model points; all except TKG4 coincide with the pits. “Site/model elevation” is the configured recorded elevation, not an independently verified ground truth. Pit and radar horizontal datum is provisionally EPSG:32608; vertical datum is unconfirmed. HRRR-AK orography varies with time, so its full forcing-period range and the corresponding temperature-correction range are shown.

| Site | Model latitude (°N) | Model longitude (°E) | Site/model elevation (m) | Pit elevation (m) | HRRR-AK elevation range (m) | TA correction range (K) | Pit collection date | Radar collection date | Pit → nearest radar sample (m) | Model → pit (m) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| TKG2 | 58.688654 | -134.382797 | 1326 | 1326 | 1228.6–1300.2 | -0.58 to -0.15 | 2021-07-15 | Unresolved (campaign: 2021-07-09–15) | 1.8 | 0.0 |
| TKG3 | 58.656386 | -134.277751 | 1203 | 1203 | 1078.6–1156.2 | -0.75 to -0.28 | 2021-07-09 | 2021-07-13 | 6.4 | 0.0 |
| C161 | 58.632957 | -134.418588 | 1483 | 1483 | 1516.2–1533.4 | +0.20 to +0.30 | 2021-07-15 | 2021-07-15 | 23.3 | 0.0 |
| TKG4 | 58.634470 | -134.237080 | 1120 | 1067 | 995.6–1055.8 | -0.75 to -0.39 | 2021-07-07 depth; 2021-07-08 SWE | 2021-07-13 | 16.0 | 8.3 |
| DG1 | 58.617588 | -134.130047 | 1018 | 1018 | 1003.4–1041.8 | -0.09 to +0.14 | 2021-07-10 | Unresolved (campaign: 2021-07-09–15) | 11.1 | 0.0 |

Distances are horizontal UTM distances to the nearest **recorded radar sample**, not a perpendicular distance to an inferred continuous track. They use all finite radar coordinates. A small distance does not establish matching elevation, acquisition date, or independent samples. The comparison uses a separate 500 m pit-centered radar footprint and reports its radius sensitivity.

TKG4 depth was observed on July 7 and density/SWE on July 8; its radar date is July 13. TKG3's archived July 9 pit date differs from the supplied Table 1 ground-truth date. TKG2 and DG1 radar dates remain unresolved; their campaign window is not an exact acquisition date.

Regenerate this table with `python sites/radar_sites/site_metadata.py` after changing site metadata.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'sites/radar_sites/prepare_radar_site_forcing.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open from inside jif_snow_modeling')
sys.path.insert(0, str(ROOT/'sites/radar_sites'))
from visualization import (SIMULATION_COLORS as colors, SIMULATION_LABELS as labels,
    SIMULATION_LINEWIDTH, ADJUSTED_LINEWIDTH, OBSERVATION_COLOR,
    OBSERVATION_MARKERS, RC_PARAMS, comparison_legend)
from matplotlib.lines import Line2D
plt.rcParams.update(RC_PARAMS)
from prepare_radar_site_forcing import load_prepared, MODEL_ROOT
from compare_model_pit_radar import SITES, load_pits, RADAR_ROOT
from pyproj import Transformer

forcing = {site: load_prepared(site) for site in SITES}
pits = load_pits()
rows = []
for site, (meta, raw, corrected) in forcing.items():
    delta = corrected.TA - raw.TA
    rows.append(dict(site=site.upper(), model_elevation_m=meta['elevation_m'],
        pit_elevation_m=pits.loc[site,'elevation_m'], slope_deg=meta['slope_angle_deg'],
        hrrr_min_elevation_m=meta['hrrr_elevation_min_m'], hrrr_max_elevation_m=meta['hrrr_elevation_max_m'],
        correction_min_K=delta.min(), correction_mean_K=delta.mean(), correction_max_K=delta.max(),
        missing_input_hours=int(raw.isna().any(axis=1).sum()), start_UTC=raw.index.min(), end_UTC=raw.index.max(),
        coordinate_source=meta['coordinate_source'], quality_flags=meta['quality_flags']))
site_summary = pd.DataFrame(rows).set_index('site')
display(site_summary.round(3).T)


## Site locations and extracted HRRR cells

Colored circles show the model coordinates at each site's configured elevation. Open squares show stored UTM extraction-cell centers; lines connect them to the model points. Open black circles show the pits; TKG4 retains a separate model location. Black radar tracks provide context without loading a basemap or downloading tiles.

In [ ]:
radar_xy = pd.read_csv(RADAR_ROOT/'ml_swe_2021.csv', usecols=['east','north'], encoding='utf-8-sig').dropna()
transform = Transformer.from_crs(4326,32608,always_xy=True)
fig, ax = plt.subplots(figsize=(9,7),layout='constrained')
points = []
for site,(meta,_,_) in forcing.items():
    x,y = transform.transform(meta['longitude'],meta['latitude'])
    points.append((site,x,y,meta))
xmin,xmax = min(p[1] for p in points)-2500,max(p[1] for p in points)+2500
ymin,ymax = min(p[2] for p in points)-2500,max(p[2] for p in points)+2500
view = radar_xy[radar_xy.east.between(xmin,xmax)&radar_xy.north.between(ymin,ymax)]
view = view.iloc[::max(1,len(view)//8000)]
ax.scatter(view.east,view.north,s=1,color=OBSERVATION_COLOR,marker=OBSERVATION_MARKERS['radar'],label='2021 radar tracks')
for site,x,y,meta in points:
    ax.plot([x,meta['grid_x_m']],[y,meta['grid_y_m']],color='0.4',lw=1)
    ax.scatter(meta['grid_x_m'],meta['grid_y_m'],marker='s',s=100,facecolors='none',edgecolors='0.35')
sc=ax.scatter([p[1] for p in points],[p[2] for p in points],c=[p[3]['elevation_m'] for p in points],
              cmap='viridis',s=130,edgecolors='black',vmin=1000,vmax=1500,zorder=4)
for site,x,y,meta in points:
    ax.annotate(f"{site.upper()} · {meta['elevation_m']:,.0f} m",(x,y),xytext=(-9,12) if site=='dg1' else (9,12),textcoords='offset points',
                ha='right' if site=='dg1' else 'left',bbox=dict(facecolor='white',edgecolor='none',alpha=.85,pad=1))
ax.scatter(pits.east,pits.north,s=240,marker=OBSERVATION_MARKERS['pit'],facecolors='none',edgecolors=OBSERVATION_COLOR,lw=1.3,label='Snow pits')
fig.colorbar(sc,ax=ax,label='Configured model elevation (m)')
ax.set(xlim=(xmin,xmax),ylim=(ymin,ymax),xlabel='UTM Zone 8N easting (m)',ylabel='UTM Zone 8N northing (m)',title='Radar pit sites and HRRR extraction cells')
ax.set_aspect('equal');ax.ticklabel_format(style='plain',useOffset=False);ax.legend(loc='lower left')
plt.show()


## Effect of the temperature correction

Monthly means use available values; the first and final months are partial. The input audit above counts missing hours. A positive correction warms HRRR-AK; a negative correction cools it. Pressure, relative humidity, precipitation, wind, and radiation are not elevation-adjusted.

In [ ]:
rows = (len(SITES)+1)//2
fig,axes=plt.subplots(rows,2,figsize=(16,4*rows),sharex=True,sharey=True)
for ax,(site,(meta,raw,corrected)) in zip(axes.flat,forcing.items()):
    for name,frame in [('hrrrak',raw),('hrrrak_lapserate',corrected)]:
        (frame.TA-273.15).resample('MS').mean().plot(ax=ax,color=colors[name],lw=SIMULATION_LINEWIDTH,label=labels[name])
    ax.axhline(0,color='0.5',lw=.7)
    ax.set(title=f"{site.upper()} · correction {meta['correction_min_K']:+.2f} to {meta['correction_max_K']:+.2f} K",ylabel='Monthly mean air temperature (°C)',xlabel='UTC')
    ax.grid(alpha=.2)
for ax in axes.flat[:len(SITES)]: ax.tick_params(axis='x',labelbottom=True)
for ax in axes.flat[len(SITES):]: ax.set_visible(False)
fig.tight_layout(rect=(0,.09,1,1))
handles=[Line2D([],[],color=colors[f],lw=SIMULATION_LINEWIDTH,label=labels[f]) for f in colors]
comparison_legend(fig,handles,columns=2,bottom=.10)
plt.show()


## Preparation and running

The notebooks for individual sites have been consolidated here. `prepare_radar_site_forcing.py` creates inputs only when they are absent; a complete existing setup is loaded and checked instead. Different requested settings or a partial existing setup cause an explanatory error rather than overwriting files.

From the repository root, run `python sites/radar_sites/prepare_radar_site_forcing.py` to prepare or reuse the inputs, then `python sites/radar_sites/run_radar_sites.py` to run all ten simulations over the same dates. The runner writes per-run logs and status JSON files under each model site's `logs/` folder. Review results in `02_compare_model_pit_radar_WY2021.ipynb`.